<a href="https://colab.research.google.com/github/jaredguerrerososa/taller_nlp/blob/main/TallerNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ¿Entienden las máquinas lo que escribimos?
### Un taller práctico de Procesamiento de Lenguaje Natural


¿Puede una máquina encontrar artículos científicos relacionados, aunque utilicen palabras diferentes? ¿Puede descubrir automáticamente qué temas investigan los científicos? ¿Y puede explicar esos temas utilizando inteligencia artificial generativa?

En este taller experimentaremos con técnicas de **Procesamiento de Lenguaje Natural (PLN)** utilizando publicaciones científicas reales de Computer Science de la Universidad de Castilla-La Mancha.

### Nuestra misión

A partir de un conjunto de publicaciones científicas, construiremos:

1. Un sistema que compare textos y mida su similitud.
2. Un buscador semántico de artículos científicos.
3. Un mapa de temas de investigación descubiertos automáticamente.
4. Un sistema que utilice Llama para interpretar esos temas.

**No necesitas instalar nada.** Trabajaremos directamente en Google Colab.



### Acceso a Llama 3.2 desde Hugging Face

Durante el taller utilizaremos **Llama 3.2 (1B Instruct)**, un modelo de lenguaje desarrollado por Meta, para generar descripciones de los temas de investigación que descubramos.

Este modelo está disponible en **Hugging Face**, una plataforma que permite acceder a modelos de inteligencia artificial y utilizarlos en nuestros propios proyectos.

Antes de comenzar, necesitamos crear una cuenta, aceptar las condiciones de uso de Llama y obtener un token de acceso. Para ello, seguid estos pasos:

1. **Crear una cuenta en Hugging Face:** acceded a [https://huggingface.co](https://huggingface.co) y registraos si todavía no tenéis una cuenta.
2. **Solicitar acceso a Llama 3.2:** entrad en [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) y aceptad las condiciones de uso del modelo. Es necesario esperar a que se autorice el acceso.
3. **Generar un token de acceso:** acceded a [Hugging Face → Access Tokens](https://huggingface.co/settings/tokens) y cread un token. En los permisos, seleccionad **User permissions → Repositories → Read access to contents of all public gated repos you can access**.

Una vez completados estos pasos, tendremos las credenciales necesarias para descargar y utilizar Llama 3.2 desde Google Colab.

**Importante:** guardad vuestro token, ya que lo necesitaremos más adelante. Es una credencial personal, por lo que no debéis compartirlo ni escribirlo directamente en el código del cuaderno. Cuando llegue el momento, lo introduciremos de forma segura.

### Preparamos las herramientas

Empezamos cargando tres bibliotecas que utilizaremos a lo largo del taller. **Pandas** nos ayudará a trabajar con las publicaciones como si fueran una tabla; **Matplotlib** y **Seaborn** nos servirán para dibujar gráficos.

Con `sns.set_theme(...)` dejamos preparado un estilo sencillo para las figuras. Si aparece el mensaje *Entorno preparado correctamente*, podemos continuar.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Entorno preparado correctamente.")


### Cargamos las publicaciones

Nuestro punto de partida es un archivo CSV con publicaciones científicas del área de Informática de la UCLM. Está alojado en un repositorio público de GitHub, así que no hace falta descargarlo ni subirlo manualmente.

`pd.read_csv(url)` lee el archivo y lo guarda en `df`, una tabla con la que trabajaremos durante todo el taller. Después comprobamos cuántas publicaciones hay, qué columnas tenemos y cómo son las primeras filas.

**Fíjate en:** `Title` contiene el título, `Year` el año y `Abstract` el resumen de cada publicación.


In [ ]:
# Dataset público alojado en GitHub
url = (
    "https://raw.githubusercontent.com/"
    "jaredguerrerososa/taller_nlp/main/computer_science_scopus.csv"
)

df = pd.read_csv(url)

print(f"\nPublicaciones cargadas: {len(df):,}")
print(f"Columnas disponibles: {list(df.columns)}")

display(df.head())


### ¿Cuándo se publicaron estos trabajos?

Antes de analizar textos, vamos a conocer un poco nuestros datos. `groupby("Year")` reúne las publicaciones por año y `size()` cuenta cuántas hay en cada grupo. `sort_index()` las ordena cronológicamente.

Con Seaborn dibujamos un gráfico de barras: cada barra representa un año y su altura indica el número de publicaciones. Al final mostramos también el total y el período que cubre el archivo.



In [ ]:

# Número de publicaciones por año
publicaciones_anio = (
    df.groupby("Year")
      .size()
      .sort_index()
)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=publicaciones_anio.index.astype(str),
    y=publicaciones_anio.values,
    color="#2A9D8F"
)

plt.title(
    "Producción científica en Computer Science - UCLM",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Año de publicación")
plt.ylabel("Número de publicaciones")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Total de publicaciones: {len(df):,}")
print(f"Período: {df['Year'].min()} - {df['Year'].max()}")


## 1. Comparación de textos y similitud

### Antes de empezar: ¿qué es el PLN?

El **Procesamiento del Lenguaje Natural (PLN)**, o *NLP* por sus siglas en inglés, es una rama de la inteligencia artificial que trabaja con el lenguaje que utilizamos las personas: textos, preguntas, noticias, conversaciones, etc.

En este taller nos centraremos en textos científicos. La idea es pasar de tener miles de títulos y resúmenes a poder **compararlos, buscarlos y organizarlos por temas**.

**Corpus:** es el conjunto de textos con el que trabajamos. Aquí, nuestro corpus está formado por las publicaciones científicas del dataset.

**Token:** es una unidad en la que dividimos un texto para procesarlo. En un ejemplo sencillo, puede ser una palabra; en modelos más avanzados, también puede ser una parte de una palabra.

**Tokenización:** es el proceso de separar un texto en esas unidades. Por ejemplo, «Machine learning improves diagnosis» puede dividirse en `Machine`, `learning`, `improves` y `diagnosis`.

**Vectorización:** significa convertir textos en listas de números. Es necesaria porque los algoritmos trabajan con números, no con frases tal como las leemos.


## ¿Cómo convertimos palabras en números?

Las personas podemos leer el título de un artículo científico e interpretar su significado. Pero ¿cómo puede una máquina comparar dos títulos?

Una estrategia consiste en representar los textos mediante números.

En esta actividad vamos a:

1. Dividir los títulos en palabras (*tokenización*).
2. Representar los títulos mediante TF-IDF.
3. Calcular la similitud entre publicaciones.
4. Visualizar los resultados mediante un mapa de calor.

**Pregunta:** ¿Dos artículos que hablan de lo mismo siempre utilizan las mismas palabras?


### Primer paso: separar las palabras de un título

Vamos a tomar el primer título del dataset con `df.iloc[0]["Title"]`. A continuación, `TfidfVectorizer` identifica los términos que aparecen en él.

La opción `stop_words="english"` descarta palabras inglesas muy frecuentes, como *the*, *and* o *of*, que normalmente aportan poca información para distinguir documentos. Con `get_feature_names_out()` mostramos los términos que ha reconocido.

Esto es una primera aproximación a la **tokenización**. Aquí no estamos interpretando el significado del título: estamos identificando sus palabras.


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Seleccionamos un título del dataset
titulo = df.iloc[0]["Title"]

# Tokenización sencilla
vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectorizer.fit([titulo])

tokens = vectorizer.get_feature_names_out()

print("Título original:")
print(titulo)

print("\nPalabras identificadas:")
print(list(tokens))


### Elegimos títulos de tres áreas distintas

Para hacer una comparación manejable, vamos a buscar títulos relacionados con **inteligencia artificial**, **computación cuántica** y **movimiento humano**.

El diccionario `areas` guarda algunas expresiones habituales de cada campo. El código recorre esas expresiones, busca coincidencias en la columna `Title` con `str.contains()` y selecciona hasta tres títulos por área, evitando repetirlos.

Al terminar veremos qué publicaciones se han elegido. **Importante:** es una selección automática basada en palabras del título, así que conviene leerlas y comprobar que tienen sentido.


In [ ]:

# Seleccionamos publicaciones de tres áreas temáticas

areas = {
    "IA": [
        "machine learning",
        "deep learning",
        "neural network"
    ],
    "Cuántica": [
        "quantum computing",
        "quantum software",
        "quantum"
    ],
      "Movimiento humano": [
        "gait",
        "posture",
        "human movement",
        "motion",
        "skeletal",
        "human activity"
  ]
}

titulos = []
etiquetas = []

for area, consultas in areas.items():

    seleccionados = []

    for consulta in consultas:

        coincidencias = df[
            df["Title"].str.contains(
                consulta,
                case=False,
                na=False,
                regex=False
            )
        ]

        for titulo in coincidencias["Title"]:
            if titulo not in titulos and titulo not in seleccionados:
                seleccionados.append(titulo)
                break

    for titulo in seleccionados[:3]:
        titulos.append(titulo)
        etiquetas.append(area)

print(f"Publicaciones seleccionadas: {len(titulos)}\n")

for i, (titulo, area) in enumerate(zip(titulos, etiquetas), 1):
    print(f"{i}. [{area}] {titulo}")


### ¿Qué es TF-IDF?

**TF-IDF** (*Term Frequency–Inverse Document Frequency*) es una forma de representar textos mediante números según las palabras que contienen.

La idea es bastante intuitiva: una palabra tiene más valor para describir un documento si **aparece en ese documento, pero no aparece en todos los demás**.

Por ejemplo, si casi todos los artículos contienen «study», esa palabra ayuda poco a distinguirlos. En cambio, «quantum» puede ser muy útil para reconocer un artículo de computación cuántica.

TF-IDF combina dos ideas:

- **TF (frecuencia del término):** cuánto aparece una palabra en un documento.
- **IDF (frecuencia inversa de documentos):** cuánto destaca esa palabra frente al resto del corpus.

También utilizamos **n-gramas**: grupos de palabras consecutivas. Un *unigrama* es una palabra, como «learning»; un *bigrama* son dos, como «machine learning». Así podemos conservar expresiones que tienen más sentido juntas.

**Palabras vacías (*stopwords*):** son palabras muy frecuentes, como «the», «and» u «of», que normalmente aportan poca información para diferenciar documentos. Por eso las excluimos en este ejemplo.

### Medimos cuánto se parecen los títulos con TF-IDF

Ahora convertimos los títulos seleccionados en números. `TfidfVectorizer` da más importancia a los términos que ayudan a distinguir un texto de los demás. Con `ngram_range=(1, 2)` tenemos en cuenta tanto palabras sueltas como parejas de palabras.

`fit_transform(titulos)` construye esa representación y `cosine_similarity(X)` calcula la similitud entre cada par de títulos. El resultado aparece en un **mapa de calor**: los valores cercanos a 1 indican mayor parecido según TF-IDF; los cercanos a 0, menor parecido.

**Pregunta:** si dos títulos hablan de lo mismo, pero usan palabras diferentes, ¿crees que TF-IDF los relacionará bien?


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(titulos)
similitud = cosine_similarity(X)

nombres = [
    f"{i+1}. {area}"
    for i, area in enumerate(etiquetas)
]

plt.figure(figsize=(10, 8))

sns.heatmap(
    similitud,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    xticklabels=nombres,
    yticklabels=nombres,
    square=True,
    linewidths=0.5
)

plt.title(
    "Similitud entre publicaciones científicas (TF-IDF)",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()


### ¿Qué entendemos por similitud?

La **similitud de coseno** compara dos vectores y nos dice si apuntan en direcciones parecidas. En nuestro caso, esos vectores representan textos.

Cuando trabajamos con vectores de valores no negativos, como los de TF-IDF, un valor cercano a **1** indica que los textos se parecen mucho según esa representación, mientras que uno cercano a **0** indica que apenas comparten características.

Ojo: **parecerse en TF-IDF no es lo mismo que significar lo mismo**. Dos frases pueden hablar del mismo tema usando palabras diferentes. Esa es precisamente la limitación que vamos a explorar ahora.


## ¿Puede una máquina reconocer el significado de un texto?

Con TF-IDF hemos representado los artículos a partir de las palabras que contienen.

Pero encontramos una limitación: dos publicaciones pueden estar relacionadas aunque utilicen vocabularios diferentes.

Los **embeddings** permiten representar textos como vectores numéricos que capturan relaciones semánticas aprendidas durante el entrenamiento de un modelo.

Vamos a utilizar un modelo preentrenado para comparar las mismas nueve publicaciones y observar qué cambia respecto a TF-IDF.

**Pregunta:** ¿Podrá el modelo reconocer relaciones que TF-IDF no detectó?


### Instalamos el modelo de embeddings

Para ir más allá de las palabras exactas utilizaremos **Sentence Transformers**, una biblioteca que permite obtener representaciones numéricas de frases y documentos.

Esta celda instala la biblioteca en Colab. Se ejecuta una vez y, cuando termine, podremos cargar el modelo en la siguiente celda.


In [ ]:

!pip -q install sentence-transformers


### ¿Qué es un embedding?

Un **embedding** es una representación numérica de un texto que intenta recoger información sobre su significado. Podemos imaginarlo como una especie de «coordenada» del texto en un espacio con muchas dimensiones.

Por ejemplo, las frases «detecting diseases from walking patterns» y «identifying health problems through gait analysis» usan palabras distintas, pero hablan de ideas relacionadas. Un buen modelo de embeddings puede situarlas cerca.

**Sentence Transformer:** es un tipo de modelo entrenado para generar embeddings de frases o textos. Aquí usamos `all-MiniLM-L6-v2`, que transforma cada texto en un vector de **384 números**.

La diferencia principal respecto a TF-IDF es que los embeddings pueden reconocer **relaciones semánticas**, es decir, parecidos de significado que no dependen solo de repetir palabras.

### Representamos los títulos mediante embeddings

Cargamos el modelo `all-MiniLM-L6-v2` y le pasamos los títulos que seleccionamos antes. En lugar de contar únicamente palabras, este modelo genera un **embedding**: una lista de números que intenta recoger información sobre el contenido del texto.

`normalize_embeddings=True` prepara los vectores para compararlos de forma sencilla. Al final mostramos cuántos embeddings se han generado y cuántos números tiene cada uno.

No hace falta entender cada número por separado: lo interesante es **comparar los vectores entre sí**.


In [ ]:

from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = modelo.encode(
    titulos,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Número de publicaciones:", len(embeddings))
print("Dimensiones de cada embedding:", embeddings.shape[1])


### Repetimos la comparación, ahora con embeddings

Volvemos a calcular la similitud entre los mismos títulos, pero esta vez utilizamos los embeddings. Así podemos comparar el nuevo mapa de calor con el de TF-IDF.

El código calcula las similitudes con `cosine_similarity(embeddings)` y las representa con la misma escala de 0 a 1. Mantener la escala facilita comparar ambos gráficos.

**Para comentar:** busca un par de títulos con pocas palabras en común. ¿Su similitud ha cambiado al utilizar embeddings? ¿Por qué podría ocurrir?


In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

similitud_semantica = cosine_similarity(embeddings)

plt.figure(figsize=(10, 8))

sns.heatmap(
    similitud_semantica,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    xticklabels=nombres,
    yticklabels=nombres,
    square=True,
    linewidths=0.5
)

plt.title(
    "Similitud entre publicaciones científicas (Embeddings)",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()


## 2. Buscador semántico de artículos científicos

### ¿Qué es la búsqueda semántica?

La **búsqueda semántica** intenta encontrar textos relacionados con la *idea* de una consulta, aunque no utilicen exactamente las mismas palabras.

El procedimiento es sencillo de entender:

1. Convertimos los resúmenes de los artículos en embeddings.
2. Convertimos nuestra pregunta en otro embedding.
3. Calculamos qué artículos tienen una representación más parecida a la pregunta.
4. Mostramos primero los resultados con mayor similitud.

Esto se llama **recuperación de información**: encontrar, dentro de una colección, los documentos más relevantes para lo que buscamos.

**Importante:** una similitud alta no demuestra que el artículo responda perfectamente a nuestra pregunta. Siempre hay que revisar el título y el resumen.


## Construimos un buscador semántico

Imagina que queremos encontrar publicaciones científicas sobre un tema, pero no conocemos sus títulos ni las palabras exactas utilizadas por sus autores.

¿Podríamos escribir una pregunta y encontrar artículos relacionados?

Vamos a construir un **buscador semántico** utilizando los embeddings de las publicaciones científicas de la UCLM.

Nuestro buscador seguirá tres pasos:

1. Convertir los resúmenes científicos en vectores.
2. Convertir nuestra consulta en otro vector.
3. Recuperar las publicaciones con mayor similitud semántica.

**Reto:** ¿Podemos encontrar artículos relacionados sin utilizar exactamente las mismas palabras que aparecen en sus títulos?


### Preparamos el buscador con todos los resúmenes

Hasta ahora solo hemos comparado unos pocos títulos. Vamos a dar el siguiente paso: construir un buscador que trabaje con **todos los resúmenes** del dataset.

Primero extraemos la columna `Abstract`. `fillna("")` sustituye los resúmenes que falten por texto vacío, para evitar errores. Después, `modelo.encode(...)` calcula un embedding para cada resumen. `batch_size=64` indica que se procesen en grupos de 64, y la barra de progreso permite ver cómo avanza el trabajo.

**Esta parte puede tardar un poco**, porque estamos procesando miles de textos. Solo necesitamos hacerlo una vez.


In [ ]:

# Preparamos los textos del buscador
documentos = df["Abstract"].fillna("").astype(str).tolist()

print(f"Publicaciones disponibles: {len(documentos):,}")
print("Generando embeddings de los resúmenes...")

embeddings_documentos = modelo.encode(
    documentos,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("\nEmbeddings generados correctamente.")
print("Dimensiones:", embeddings_documentos.shape)


### Creamos nuestra función de búsqueda semántica

La función `buscar_articulos()` recibirá una consulta escrita en lenguaje natural y devolverá los artículos más relacionados.

Primero convertimos la consulta en un embedding. Después la comparamos con los embeddings de los resúmenes usando `@`, que aquí permite calcular rápidamente las similitudes. `np.argsort(...)` ordena los resultados y `top_k` decide cuántos queremos mostrar.

La función devuelve una tabla con el **título**, el **año**, el **resumen** y una **puntuación de similitud**. Recuerda: esa puntuación indica parecido entre vectores, no garantiza que un artículo responda perfectamente a la consulta.


In [ ]:

import numpy as np

def buscar_articulos(consulta, top_k=5):

    # Convertimos la consulta en un embedding
    embedding_consulta = modelo.encode(
        consulta,
        normalize_embeddings=True
    )

    # Calculamos la similitud con todos los artículos
    similitudes = embeddings_documentos @ embedding_consulta

    # Recuperamos los artículos más similares
    indices = np.argsort(similitudes)[::-1][:top_k]

    resultados = df.iloc[indices][
        ["Title", "Year", "Abstract"]
    ].copy()

    resultados.insert(
        0,
        "Similitud",
        np.round(similitudes[indices], 3)
    )

    return resultados.reset_index(drop=True)


### Probamos el buscador

Vamos a buscar publicaciones sobre aplicaciones de la inteligencia artificial en salud. La variable `consulta` guarda nuestra pregunta en inglés y `buscar_articulos(consulta, top_k=5)` recupera los cinco resultados más próximos.

Por ahora mostramos una tabla resumida con la similitud, el título y el año.

**Prueba después:** cambia la consulta por otro tema que te interese. No necesitas conocer el título exacto de ningún artículo.


In [ ]:

consulta = "Artificial intelligence for healthcare applications"

print("Consulta:", consulta)

resultados = buscar_articulos(consulta, top_k=5)

display(
    resultados[["Similitud", "Title", "Year"]]
)


### Leemos los resultados con más detalle

Una tabla de títulos no siempre basta para decidir si un artículo es relevante. Este bloque recorre los resultados con `iterrows()` y muestra cada publicación por separado.

Además del título, el año y la similitud, imprimimos los primeros **500 caracteres del resumen** con `fila['Abstract'][:500]`. Así podemos hacernos una idea del contenido sin leer todo el texto.

**Fíjate en:** ¿el artículo mejor puntuado es también el que tú elegirías?


In [ ]:

for i, fila in resultados.iterrows():
    print(f"\n{'=' * 70}")
    print(f"RESULTADO {i + 1}")
    print(f"{'=' * 70}")
    print(f"Título: {fila['Title']}")
    print(f"Año: {fila['Year']}")
    print(f"Similitud: {fila['Similitud']:.3f}")
    print(f"\nResumen: {fila['Abstract'][:500]}...")


### Guardamos los embeddings calculados

Calcular los embeddings de miles de resúmenes lleva trabajo, así que puede venir bien conservarlos. `np.save(...)` guarda la matriz en un archivo llamado `embeddings_uclm.npy` dentro del entorno de Colab.

Este archivo contiene los vectores numéricos, **no los artículos originales**. Nos puede servir para reutilizar el cálculo más adelante mientras tengamos acceso al archivo. El código también muestra sus dimensiones para comprobar que se ha guardado la matriz esperada.


In [ ]:

import numpy as np

np.save(
    "embeddings_uclm.npy",
    embeddings_documentos
)

print("Embeddings guardados correctamente.")
print("Dimensiones:", embeddings_documentos.shape)


### Convertimos el buscador en una pequeña aplicación

Vamos a hacer la búsqueda más cómoda con `ipywidgets`. En lugar de cambiar una variable en el código, tendremos una **caja de texto** y un **botón**.

`widgets.Text` crea el campo donde escribiremos la consulta; `widgets.Button` crea el botón; y `widgets.Output` reserva un espacio para los resultados. La función `ejecutar_busqueda()` recoge lo que escribimos, llama a `buscar_articulos()` y muestra los cinco artículos encontrados. Finalmente, `boton.on_click(...)` conecta el botón con esa función.

**Tu turno:** prueba dos consultas parecidas, pero escritas con palabras diferentes. ¿Recuperan los mismos artículos?


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

entrada = widgets.Text(
    value="Artificial intelligence for healthcare applications",
    placeholder="Escribe un tema de investigación...",
    description="Consulta:",
    layout=widgets.Layout(width="85%"),
    style={"description_width": "initial"}
)

boton = widgets.Button(
    description="Buscar artículos",
    button_style="primary",
    icon="search"
)

salida = widgets.Output()

def ejecutar_busqueda(b):
    with salida:
        clear_output(wait=True)

        consulta = entrada.value.strip()

        if not consulta:
            print("Introduce una consulta.")
            return

        resultados = buscar_articulos(consulta, top_k=5)

        print(f"Resultados para: {consulta}\n")

        for i, fila in resultados.iterrows():
            print(f"{i + 1}. {fila['Title']}")
            print(f"   Año: {fila['Year']}")
            print(f"   Similitud: {fila['Similitud']:.3f}\n")

boton.on_click(ejecutar_busqueda)

display(entrada, boton, salida)


## 3. Descubrimiento automático de temas de investigación

### ¿Qué es el modelado de temas?

El **modelado de temas** (*topic modeling*) consiste en buscar grupos de documentos que tratan asuntos parecidos sin tener que etiquetarlos uno por uno.

**BERTopic** es una herramienta que combina representaciones semánticas de los textos con técnicas de agrupamiento para descubrir esos temas. Después identifica palabras representativas que nos ayudan a entender de qué trata cada grupo.

Tres conceptos que aparecerán en el código:

- **UMAP:** reduce la cantidad de dimensiones de los embeddings, intentando conservar relaciones entre documentos cercanos. Esto facilita su agrupamiento y visualización.
- **HDBSCAN:** es un algoritmo de *clustering* (agrupamiento) que reúne documentos similares. También puede dejar algunos sin asignar si no encajan claramente en ningún grupo.
- **c-TF-IDF:** es una adaptación de TF-IDF que ayuda a encontrar las palabras más características de cada tema, considerando los documentos agrupados en él.

**Tema `-1`:** no significa que exista un «tema menos uno». Es la etiqueta que BERTopic utiliza para los documentos que HDBSCAN no ha podido asignar con suficiente claridad a un grupo.


## ¿Puede una máquina descubrir temas de investigación?

Hasta ahora hemos utilizado embeddings para comparar publicaciones y construir un buscador semántico.

Pero ¿qué sucede si no sabemos qué buscar?

Vamos a utilizar **BERTopic**, una técnica de modelado de temas que combina representaciones semánticas, agrupamiento y análisis del vocabulario.

Nuestro objetivo es descubrir automáticamente grupos de publicaciones relacionadas y explorar qué palabras caracterizan cada grupo.

**Pregunta:** ¿Qué líneas de investigación podemos identificar sin definir previamente las categorías?


### Instalamos BERTopic

Ahora cambiaremos de pregunta. Hasta aquí hemos buscado artículos a partir de una consulta; con **BERTopic** intentaremos descubrir qué temas aparecen en la colección **sin indicar previamente cuáles son**.

Esta celda instala la biblioteca que utilizaremos para agrupar documentos y obtener palabras representativas de cada tema.


In [ ]:

!pip -q install bertopic


### Preparamos los documentos para BERTopic

Utilizaremos **todas las publicaciones** del dataset. `df.copy()` crea una copia de la tabla y, a partir de ella, obtenemos los resúmenes que analizará BERTopic.

También recuperamos los embeddings que ya calculamos para el buscador. Así evitamos repetir un cálculo costoso. Los índices de `muestra` nos permiten mantener cada resumen emparejado con su embedding.

El código muestra cuántos documentos entrarán en el análisis, el tamaño de sus vectores y los años que cubren. Verás también una selección antigua de 500 documentos comentada: **no se ejecuta**.


In [ ]:

# Seleccionamos una muestra reproducible
"""muestra = df.sample(
    n=500,
    random_state=42
).sort_index()"""

muestra = df.copy()

# Textos de la muestra
documentos_bertopic = muestra["Abstract"].astype(str).tolist()

# Recuperamos sus embeddings ya calculados
embeddings_bertopic = embeddings_documentos[muestra.index.to_numpy()]

print("Documentos seleccionados:", len(documentos_bertopic))
print("Dimensiones de embeddings:", embeddings_bertopic.shape)
print("Período:", muestra["Year"].min(), "-", muestra["Year"].max())


### Dejamos que BERTopic encuentre los temas

Aquí ocurre la parte central del modelado de temas. BERTopic combina varias herramientas:

- **UMAP** reduce las dimensiones de los embeddings para facilitar el agrupamiento.
- **HDBSCAN** busca grupos de documentos parecidos y puede dejar algunos sin grupo.
- **CountVectorizer** extrae palabras y expresiones frecuentes de los textos, quitando palabras vacías en inglés.

Con `BERTopic(...)` reunimos estas piezas y con `fit_transform(...)` analizamos los resúmenes. El resultado incluye el tema asignado a cada publicación. El identificador **`-1`** corresponde a documentos que no encajan claramente en un grupo.

Como ahora utilizamos toda la colección, esta celda puede tardar más que las anteriores.


In [ ]:

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(
    documentos_bertopic,
    embeddings_bertopic
)

print("\nEntrenamiento completado.")
print("Temas identificados:", len(set(topics) - {-1}))
print("Documentos sin tema asignado:", topics.count(-1))


### ¿Cuántos temas ha encontrado?

`get_topic_info()` prepara una tabla resumen con los temas identificados por BERTopic. La columna `Topic` contiene el número del tema, `Count` indica cuántas publicaciones agrupa y `Name` muestra un nombre provisional construido a partir de palabras representativas.

Mostramos las primeras 15 filas para hacernos una idea de los resultados. **Estos nombres todavía no son etiquetas redactadas por Llama**; son una primera descripción automática.


In [ ]:

informacion_temas = topic_model.get_topic_info()

display(
    informacion_temas[
        ["Topic", "Count", "Name"]
    ].head(15)
)


### Exploramos los temas descubiertos

BERTopic ha identificado automáticamente diferentes grupos de publicaciones científicas.

Ahora vamos a visualizar:

- Qué temas tienen más publicaciones.
- Qué palabras caracterizan cada tema.
- Qué relaciones existen entre los temas.

**Pregunta:** ¿Podemos interpretar un tema científico utilizando únicamente sus palabras más representativas?


### Vemos las palabras que definen cada tema

El gráfico de barras muestra las palabras o expresiones más representativas de varios temas. `top_n_topics=12` limita la vista a doce temas y `n_words=6` enseña seis términos por cada uno.

Las barras ayudan a identificar de qué podría tratar cada grupo. No obstante, **una lista de palabras no siempre cuenta toda la historia**: para entender un tema también conviene revisar sus publicaciones.


In [ ]:

fig = topic_model.visualize_barchart(
    top_n_topics=12,
    n_words=6,
    width=300,
    height=250
)

fig.show()


### Exploramos cómo se relacionan los temas

`visualize_topics()` crea un mapa interactivo en el que cada círculo representa un tema. Su tamaño y su posición nos ayudan a explorar los grupos que ha encontrado BERTopic.

Podemos acercarnos, alejarnos y pasar el cursor por los círculos para consultar información. **Recuerda:** es una representación visual simplificada, no un mapa exacto de todas las relaciones entre artículos.


In [ ]:

fig = topic_model.visualize_topics()

fig.show()


### ¿Qué temas agrupan más publicaciones?

Vamos a contar cuántos documentos tiene cada tema. Primero apartamos el grupo `-1`, porque reúne publicaciones que BERTopic no asignó a un tema concreto.

Después utilizamos un gráfico de barras para comparar el tamaño de los grupos. Esto nos permite detectar temas muy presentes en la colección y otros más pequeños.

**Pregunta:** ¿un tema con más publicaciones tiene que ser necesariamente más importante?


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Excluimos el grupo de documentos no asignados (-1)
temas_validos = informacion_temas[
    informacion_temas["Topic"] != -1
].copy()

plt.figure(figsize=(11, 6))

sns.barplot(
    data=temas_validos.head(12),
    x="Count",
    y="Name",
    color="#2A9D8F"
)

plt.title(
    "Temas de investigación más frecuentes",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Número de publicaciones")
plt.ylabel("Tema identificado por BERTopic")

plt.tight_layout()
plt.show()


### Miramos dentro de algunos temas

Los gráficos ofrecen una visión general, pero también necesitamos ejemplos concretos. Este bloque recorre los temas **0, 3 y 5** y muestra sus palabras clave y algunos títulos representativos.

La idea es comprobar si los documentos de cada grupo realmente hablan de asuntos parecidos. Si un tema mezcla trabajos muy distintos, también es un resultado interesante: los modelos de agrupamiento no siempre trazan fronteras perfectas.

**Ten en cuenta:** al utilizar todas las publicaciones, los números de los temas pueden cambiar respecto a pruebas anteriores.


In [ ]:

for tema in [0, 3, 5]:

    print(f"\n{'=' * 70}")
    print(f"TEMA {tema}")
    print(f"{'=' * 70}")

    print("Palabras clave:")
    print(topic_model.get_topic(tema)[:8])

    print("\nEjemplos de publicaciones:")

    indices = [
        i for i, t in enumerate(topics)
        if t == tema
    ][:3]

    for i in indices:
        print(f"\n- {muestra.iloc[i]['Title']}")


## 4. Interpretación de temas mediante Llama

### ¿Qué aporta un modelo de lenguaje como Llama?

Un **modelo de lenguaje** aprende patrones a partir de grandes cantidades de texto y puede generar respuestas a partir de instrucciones.

En esta parte utilizamos **Llama 3.2** para poner nombres más fáciles de leer a los temas encontrados por BERTopic.

Es importante distinguir sus funciones:

- **BERTopic descubre los grupos** de artículos y sus palabras representativas.
- **Llama propone una etiqueta** a partir de esa información.

**Prompt:** es el texto con el que damos instrucciones al modelo. En nuestro caso, le indicamos cómo debe nombrar los temas.

**Few-shot prompting:** consiste en incluir uno o varios ejemplos dentro del prompt para mostrarle al modelo el tipo de respuesta que esperamos.

Una etiqueta generada puede sonar convincente y aun así ser incorrecta o demasiado específica. Por eso **hay que contrastarla con las palabras clave y los artículos reales**.



### ¿Puede una IA generativa explicar los temas descubiertos?

BERTopic ha identificado grupos de publicaciones y sus palabras más representativas.

Sin embargo, nombres como `model_surface_parameters_method` no siempre son fáciles de interpretar.

Vamos a utilizar **Llama**, un modelo de lenguaje generativo, para proponer etiquetas más comprensibles.

Para ello, le proporcionaremos:

- Las palabras representativas de cada tema.
- Ejemplos reales de títulos científicos.
- Instrucciones para generar una etiqueta y una breve explicación.

**Importante:** Llama no descubrirá nuevos temas. Su función será interpretar los grupos que BERTopic ya ha identificado.

**Pregunta:** ¿Debemos confiar automáticamente en las etiquetas propuestas por un modelo generativo?

### Comprobamos si tenemos GPU

Para la siguiente parte utilizaremos un modelo generativo llamado **Llama**. Antes de cargarlo, comprobamos si Colab tiene una GPU disponible.

`torch.cuda.is_available()` nos dice si PyTorch puede utilizarla. Si la respuesta es positiva, mostramos el nombre de la GPU. Esto importa porque generar texto con un modelo de lenguaje suele ser mucho más rápido en GPU que en CPU.


In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Modelo de GPU:", torch.cuda.get_device_name(0))
    memoria = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Memoria total: {memoria:.1f} GB")
else:
    print("No se ha detectado GPU.")


### Introducimos nuestro token de Hugging Face

El modelo oficial que vamos a utilizar requiere una cuenta de Hugging Face con acceso autorizado. `getpass()` permite introducir el token sin mostrarlo en pantalla ni escribirlo directamente en el código del cuaderno.

**Importante:** cada participante debe utilizar su propio token y contar con acceso al repositorio del modelo. No compartas el token con otras personas ni lo incluyas al publicar el notebook.


In [ ]:
from getpass import getpass

token_hf = getpass("Introduce tu token de Hugging Face: ")


### Comprobamos el permiso para utilizar Llama

Antes de descargar el modelo completo, comprobamos que nuestro token puede acceder a su repositorio. `hf_hub_download()` solicita un archivo pequeño llamado `config.json`.

Si todo va bien, veremos un mensaje de acceso confirmado. Si falla, el código mostrará el tipo de error para ayudarnos a distinguir un problema de permisos de otros fallos de conexión.

Esta comprobación evita iniciar una descarga grande cuando todavía no tenemos autorización.


In [ ]:

from huggingface_hub import hf_hub_download

modelo_llama_id = "meta-llama/Llama-3.2-1B-Instruct"

try:
    hf_hub_download(
        repo_id=modelo_llama_id,
        filename="config.json",
        token=token_hf,
        force_download=True
    )
    print("✅ Acceso confirmado a Llama 3.2 1B Instruct.")

except Exception as e:
    print("❌ Acceso no confirmado.")
    print("Tipo de error:", type(e).__name__)
    print("Detalle:", str(e)[:600])


### Cargamos Llama 3.2

Ahora sí, cargamos **Llama 3.2 1B Instruct**, un modelo generativo que utilizaremos para proponer nombres más fáciles de leer para los temas de BERTopic.

`AutoTokenizer` prepara los textos para el modelo y `AutoModelForCausalLM` carga el modelo propiamente dicho. `dtype=torch.float16` ayuda a reducir el consumo de memoria y `device_map="auto"` permite elegir el dispositivo disponible. Con `eval()` lo dejamos listo para generar respuestas.

La primera carga puede tardar porque necesita descargar los archivos del modelo. Al final comprobamos el dispositivo y el número aproximado de parámetros.


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo_llama_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer_llama = AutoTokenizer.from_pretrained(
    modelo_llama_id,
    token=token_hf
)

modelo_llama = AutoModelForCausalLM.from_pretrained(
    modelo_llama_id,
    token=token_hf,
    dtype=torch.float16,
    device_map="auto"
)

modelo_llama.eval()

print("Llama 3.2 cargado correctamente.")
print("Dispositivo:", modelo_llama.device)
print(
    "Parámetros:",
    f"{sum(p.numel() for p in modelo_llama.parameters()) / 1e9:.2f}B"
)


### Preparamos la generación de texto

La función `pipeline()` nos da una forma cómoda de pedirle texto a Llama. Aquí reutilizamos el modelo y el tokenizador que acabamos de cargar.

`max_new_tokens=40` limita la longitud de las respuestas: queremos **etiquetas cortas**, no párrafos. `do_sample=False` evita introducir muestreo aleatorio en la generación y `repetition_penalty` ayuda a reducir repeticiones.

Este generador será la pieza que conectaremos con BERTopic.

In [ ]:

from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model=modelo_llama,
    tokenizer=tokenizer_llama,
    max_new_tokens=40,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False
)

print("Generador de Llama 2 preparado.")


### Le enseñamos a Llama cómo queremos las etiquetas

Antes de pedirle que nombre nuestros temas, vamos a explicarle la tarea mediante un **prompt**, es decir, unas instrucciones.

`system_prompt` le indica que debe crear etiquetas breves para temas científicos. `example_prompt` presenta un ejemplo sobre alimentación y medioambiente, junto con una etiqueta adecuada. `main_prompt` contiene dos marcadores, `[DOCUMENTS]` y `[KEYWORDS]`, que después se rellenarán con documentos y palabras de cada tema.

La lista `mensajes` reúne las instrucciones, el ejemplo y la petición. `apply_chat_template()` las adapta al formato que espera Llama 3.2.

**La idea:** en vez de esperar que el modelo adivine qué formato queremos, le mostramos un ejemplo claro.


In [ ]:

system_prompt = (
    "You are a scientific topic labeling assistant. "
    "Create short, accurate labels for research topics. "
    "Return only the label, without explanations."
)

example_prompt = """
I have a topic that contains the following documents:
- Traditional diets in most cultures were primarily plant-based.
- Meat, especially beef, has a high environmental impact.
- Eating less meat can reduce greenhouse gas emissions.

The topic is described by the following keywords:
meat, beef, eating, emissions, food, health

Create a short label for this topic.
"""

main_prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords:
[KEYWORDS]

Create a short, accurate label for this topic.
Return only the label.
"""

mensajes = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example_prompt},
    {
        "role": "assistant",
        "content": "Environmental Impact of Meat Consumption"
    },
    {"role": "user", "content": main_prompt}
]

prompt = tokenizer_llama.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True
)

print("Prompt para Llama 3.2 preparado.")


### Conectamos Llama con BERTopic

La clase `TextGeneration` permite utilizar un generador de texto como parte de la representación de temas de BERTopic.

Le pasamos el generador y el prompt que acabamos de preparar. `nr_docs=3` indica que se incluirán tres documentos como ejemplos del tema, mientras que `diversity=0.3` ayuda a seleccionar ejemplos variados.

Guardamos esta configuración en `representation_model`, bajo el nombre **Llama**. Todavía no hemos cambiado los temas: estamos preparando la herramienta que propondrá sus etiquetas.


In [ ]:

from bertopic.representation import TextGeneration

llama = TextGeneration(
    generator,
    prompt=prompt,
    nr_docs=3,
    diversity=0.3
)

representation_model = {
    "Llama": llama
}

print("Representación con Llama preparada.")


### Generamos etiquetas más fáciles de interpretar

Ahora pedimos a BERTopic que actualice la descripción de sus temas. El `CountVectorizer` vuelve a preparar las palabras representativas, eliminando palabras vacías en inglés y teniendo en cuenta expresiones de una o dos palabras.

Después, `update_topics()` utiliza esas palabras y los documentos de cada tema para solicitar una etiqueta a Llama. **No vuelve a ejecutar el agrupamiento de UMAP y HDBSCAN**; actualiza la forma de describir los temas existentes.

Esta celda puede tardar, porque Llama tiene que generar varias respuestas.


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

topic_model.update_topics(
    documentos_bertopic,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print("Palabras clave y etiquetas actualizadas.")


### Consultamos las etiquetas generadas

Volvemos a pedir la tabla de temas con `get_topic_info()`. Ahora aparece una columna nueva, `Llama`, con las etiquetas propuestas por el modelo generativo.

Podemos comparar el nombre provisional de BERTopic (`Name`) con el nombre sugerido por Llama. **No demos por hecho que la etiqueta es correcta solo porque suena bien**: más adelante la contrastaremos con las publicaciones.


In [ ]:

informacion_temas = topic_model.get_topic_info()

display(
    informacion_temas[
        ["Topic", "Count", "Name", "Llama"]
    ]
)


### Ponemos las palabras clave y las etiquetas una al lado de la otra

Para facilitar la comparación, preparamos una tabla más limpia. Quitamos el grupo `-1`, extraemos la primera etiqueta de la lista que devuelve Llama y reunimos las ocho palabras clave principales de cada tema.

El resultado muestra, para cada tema, cuántas publicaciones contiene, qué palabras lo describen y qué nombre propone Llama.

**Para comentar:** ¿hay etiquetas que parecen demasiado concretas, demasiado generales o poco relacionadas con las palabras clave?


In [ ]:

import pandas as pd

informacion_temas = topic_model.get_topic_info()

comparacion = informacion_temas[
    informacion_temas["Topic"] != -1
].copy()

comparacion["Etiqueta_Llama"] = comparacion["Llama"].apply(
    lambda etiquetas: etiquetas[0] if etiquetas else "Sin etiqueta"
)

comparacion["Palabras_clave"] = comparacion["Topic"].apply(
    lambda tema: ", ".join(
        palabra for palabra, peso in topic_model.get_topic(tema)[:8]
    )
)

display(
    comparacion[
        ["Topic", "Count", "Palabras_clave", "Etiqueta_Llama"]
    ]
)


### Usamos las etiquetas de Llama en los gráficos

Hasta ahora, las etiquetas estaban en una tabla. Con `set_topic_labels()` se las asignamos a BERTopic para que también aparezcan en sus visualizaciones.

El diccionario `etiquetas_llama` relaciona cada número de tema con su etiqueta. Dejamos fuera el grupo `-1`, porque no corresponde a un tema bien definido.

Así, al explorar el mapa, veremos nombres más comprensibles que una simple lista de palabras.


In [ ]:
etiquetas_llama = {
    int(fila["Topic"]): fila["Llama"][0]
    for _, fila in topic_model.get_topic_info().iterrows()
    if fila["Topic"] != -1
}

topic_model.set_topic_labels(etiquetas_llama)

print("Etiquetas de Llama asignadas correctamente.")

### ¿Por qué dibujamos un mapa en dos dimensiones?

Los embeddings de nuestro modelo tienen **384 dimensiones**, así que no podemos dibujarlos directamente en una gráfica normal.

Usamos **reducción de dimensionalidad** para obtener una representación en dos dimensiones: un eje horizontal y otro vertical. De esta forma podemos explorar visualmente qué publicaciones quedan cerca.

El mapa es una **aproximación visual**, no una medida exacta de todas las distancias originales. Dos puntos cercanos suelen compartir características, pero no debemos sacar conclusiones únicamente por su posición.

### Preparamos un mapa de las publicaciones

Cada resumen está representado por un embedding con muchas dimensiones. Para poder dibujar los documentos en una pantalla, utilizamos **UMAP** y reducimos esos vectores a **dos dimensiones**.

`fit_transform(embeddings_bertopic)` calcula las posiciones que usaremos en el gráfico. El resultado tendrá dos números por publicación: una coordenada horizontal y otra vertical.

**Importante:** las distancias del mapa son una aproximación visual. No debemos interpretarlas como medidas exactas de parecido.


In [ ]:
from umap import UMAP

umap_visualizacion = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

reduced_embeddings = umap_visualizacion.fit_transform(
    embeddings_bertopic
)

print("Dimensiones:", reduced_embeddings.shape)

### Exploramos las publicaciones en un mapa interactivo

Llegamos a una de las visualizaciones más completas del taller. `visualize_documents()` dibuja las publicaciones utilizando las coordenadas que acabamos de calcular y los temas identificados por BERTopic.

En lugar de mostrar los resúmenes completos, utilizamos los **títulos** para que sea más cómodo explorar los puntos. Las etiquetas personalizadas (`custom_labels=True`) son las que propuso Llama. Podemos pasar el cursor por las publicaciones y comprobar si encajan en el tema asignado.

**Prueba:** elige un grupo y lee varios títulos. ¿Te parece acertada la etiqueta?


In [ ]:
titulos_bertopic = (
    muestra["Title"].fillna("Sin título").astype(str).tolist()
)

fig = topic_model.visualize_documents(
    titulos_bertopic,
    reduced_embeddings=reduced_embeddings,
    hide_annotations=True,
    hide_document_hover=False,
    custom_labels=True,
    title="Mapa de publicaciones científicas - BERTopic + Llama 3.2"
)

fig.show()